In [1]:
!python3 --version

Python 3.10.11


In [2]:
!pip3 install faiss-cpu sentence-transformers transformers


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
from sentence_transformers import SentenceTransformer

# Cargar modelo localmente
embedder = SentenceTransformer("../modelos/all-MiniLM-L6-v2")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import os, logging

# Logging mínimo
logging.basicConfig(
    filename="../logs/03_EMBEDING.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [3]:
import pandas as pd

df = pd.read_csv("../data/processed/productos_corpus.csv",delimiter=",")
logging.info("Se carga el corpus csv")

In [4]:
df['CHUNK_DESCRIPCION'] = df['CHUNK_DESCRIPCION'].fillna('')
df['CHUNK_CARACTERISTICAS'] = df['CHUNK_CARACTERISTICAS'].fillna('')
df['CHUNK_OBSERVACIONES'] = df['CHUNK_OBSERVACIONES'].fillna('')
df['CHUNK_RECOMENDACIONES'] = df['CHUNK_RECOMENDACIONES'].fillna('')
logging.info("Se remplaza los chunk NULL por cadena vacia.")

In [5]:
import numpy as np

# Convertir textos a embeddings
corpus_embeddings = embedder.encode(df['CHUNK_PRODUCTO'].tolist(), convert_to_numpy=True)
np.save("../embeddings/CHUNK_PRODUCTO.npy", corpus_embeddings)
logging.info("Se guarda el embeding de Producto")

corpus_embeddings = embedder.encode(df['CHUNK_FICHA'].tolist(), convert_to_numpy=True)
np.save("../embeddings/CHUNK_FICHA.npy", corpus_embeddings)
logging.info("Se guarda el embeding de Ficha")

corpus_embeddings = embedder.encode(df['CHUNK_DESCRIPCION'].tolist(), convert_to_numpy=True)
np.save("../embeddings/CHUNK_DESCRIPCION.npy", corpus_embeddings)
logging.info("Se guarda el embeding de Descripcion")

corpus_embeddings = embedder.encode(df['CHUNK_CARACTERISTICAS'].tolist(), convert_to_numpy=True)
np.save("../embeddings/CHUNK_CARACTERISTICAS.npy", corpus_embeddings)
logging.info("Se guarda el embeding de Caracteristica")

corpus_embeddings = embedder.encode(df['CHUNK_OBSERVACIONES'].tolist(), convert_to_numpy=True)
np.save("../embeddings/CHUNK_OBSERVACIONES.npy", corpus_embeddings)
logging.info("Se guarda el embeding de Observaciones")

corpus_embeddings = embedder.encode(df['CHUNK_RECOMENDACIONES'].tolist(), convert_to_numpy=True)
np.save("../embeddings/CHUNK_RECOMENDACIONES.npy", corpus_embeddings)
logging.info("Se guarda el embeding de Recomendaciones")

Batches: 100%|██████████| 1453/1453 [00:40<00:00, 35.99it/s] 


In [7]:
import numpy as np

embeddings = np.load("../embeddings/CHUNK_PRODUCTO.npy")
len(embeddings)

46490

In [10]:
import numpy as np

logging.info("Inicio de concatenacion de embedings")
embeddings = np.load("../embeddings/CHUNK_PRODUCTO.npy")
embeddings = np.concatenate((embeddings,np.load("../embeddings/CHUNK_FICHA.npy")))
embeddings = np.concatenate((embeddings,np.load("../embeddings/CHUNK_DESCRIPCION.npy")))
embeddings = np.concatenate((embeddings,np.load("../embeddings/CHUNK_CARACTERISTICAS.npy")))
embeddings = np.concatenate((embeddings,np.load("../embeddings/CHUNK_OBSERVACIONES.npy")))
embeddings = np.concatenate((embeddings,np.load("../embeddings/CHUNK_RECOMENDACIONES.npy")))

len(embeddings)
logging.info("Fin de concatenacion de embedings, total: "+str(len(embeddings)))

In [11]:
import faiss

# Crear índice FAISS
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [12]:
def responder_pregunta_rag(pregunta, k=10):
    # Embed la pregunta
    pregunta_emb = embedder.encode([pregunta], convert_to_numpy=True)

    # Buscar los k textos más cercanos
    distancias, indices = index.search(pregunta_emb, k)
    print(distancias)
    print(indices)

responder_pregunta_rag('necesito una refrigeradora')

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.57s/it]

[[0.47783184 0.48447216 0.48447216 0.48447216 0.48447216 0.48447216
  0.48447216 0.48447216 0.48447216 0.52488136]]
[[176543 224699 224700 224701 224702 224703 224704 227696 227697 227731]]


In [13]:
responder_pregunta_rag('necesito una refrigeradora')

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.85it/s]

[[0.47783184 0.48447216 0.48447216 0.48447216 0.48447216 0.48447216
  0.48447216 0.48447216 0.48447216 0.52488136]]
[[176543 224699 224700 224701 224702 224703 224704 227696 227697 227731]]


In [11]:
#resto = 256196 % 46490
cociente, resto = divmod(176543, 46490)
print(cociente)
print(resto)
# ['CHUNK_PRODUCTO', 'CHUNK_FICHA', 'CHUNK_DESCRIPCION', 'CHUNK_CARACTERISTICAS', 'CHUNK_OBSERVACIONES', 'CHUNK_RECOMENDACIONES']
# 46490 productos del 0 al 46489

3
37073


In [12]:
df.iloc[37073]

SKU                                                                 156424
DESCRIPCION                        OSTER FRIGOBAR 122LTS OS-PMB129BB NEGRO
DES_DIV                                                       HOGAR Y DECO
DES_AREA                                                     ELECTRO HOGAR
DES_DPTO                                                      LINEA BLANCA
DES_LIN                                                     REFRIGERADORAS
DES_MARCA                                                            OSTER
DES_PROCE                                                         NACIONAL
URL                                                                    NaN
CHUNK_PRODUCTO           oster frigobar 122lts os-pmb129bb negro de la ...
CHUNK_FICHA              acabado liso, altura del producto 84.00 cm, an...
CHUNK_DESCRIPCION        Compartimiento refrigerado. Control de tempera...
CHUNK_CARACTERISTICAS    Compartimiento refrigerado. Patas ajustables. ...
CHUNK_OBSERVACIONES      